# 03 · Comment embeddings

Interactive maps of the Sentence-BERT embeddings (`all-mpnet-base-v2`, from `scripts/data/embed_comments.py`).
**Hover over a dot to read the comment.**

- A random 10,000 of the 38,509 comments, plus every comment in the train and test consensus samples.
- 2D layout by UMAP (cosine). Nearby dots = similar meaning; the axes have no units.
- The layout is computed once and cached in `data/processed/civil-comments/comments/embeddings/umap.parquet`
  (delete it to recompute, ~20 s).

> **Content warning:** the comments include toxic language.

In [20]:
import textwrap

import numpy as np
import pandas as pd
import plotly.express as px

from ethos.data import load_embeddings
from ethos.paths import COMMENTS, EMBEDDINGS, TEST_SAMPLE, TRAIN_SAMPLE

N_BACKGROUND = 10_000
emb = load_embeddings()
comments = pd.read_csv(COMMENTS / "comments.csv",
                       usecols=["id", "comment_text", "toxicity", "consensus", "split"]).set_index("id")
in_sample = {**{i: "train sample" for i in pd.read_csv(TRAIN_SAMPLE / "sample.csv").id},
             **{i: "test sample" for i in pd.read_csv(TEST_SAMPLE / "sample.csv").id}}
print(f"{len(emb):,} comments x {emb.shape[1]} dimensions")

38,509 comments x 768 dimensions


## 2D layout (UMAP)

In [21]:
cache = EMBEDDINGS / "umap.parquet"
if cache.exists():
    xy = pd.read_parquet(cache)
else:
    import umap
    rng = np.random.default_rng(0)
    others = emb.index.difference(list(in_sample))
    ids = pd.Index(list(in_sample)).append(pd.Index(rng.choice(others, N_BACKGROUND, replace=False)))
    coords = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0).fit_transform(emb.loc[ids].to_numpy())
    xy = pd.DataFrame(coords, index=ids, columns=["umap_1", "umap_2"])
    xy.to_parquet(cache)
    
xy.index.name = "id"

def wrap(text, width=70, max_chars=500):
    """Comment text for a hover box: whitespace tidied, cut to max_chars, wrapped with <br>."""
    text = " ".join(str(text).split())
    text = text if len(text) <= max_chars else text[:max_chars].rsplit(" ", 1)[0] + " [...]"
    return "<br>".join(textwrap.wrap(text, width))

df = xy.join(comments)
df["comment"] = df.comment_text.map(wrap)
df["sample"] = df.index.map(lambda i: in_sample.get(i, "other"))
df["consensus band"] = pd.cut(df.consensus, [0.5, 0.6, 0.7, 0.8, 0.9, 1.0001], right=False,
                              labels=["0.5-0.6", "0.6-0.7", "0.7-0.8", "0.8-0.9", "0.9-1"]).astype(str)
hover = {"comment": True, "toxicity": ":.2f", "consensus": ":.2f", "split": True, "sample": True,
         "umap_1": False, "umap_2": False}
print(f"{len(df):,} comments on the map")

10,250 comments on the map


## Topics

k-means groups all 38,509 comments into `N_TOPICS` clusters of similar meaning (on the full 768-dim embeddings, not the
2D map). Topics are numbered from largest to smallest.

Each topic is described by its 3 most distinctive words, chosen automatically: words used in at least 20 comments
(common English filler words removed), scored by `share of the topic's comments using the word × log(that share / share
of all comments using it)`, so a word scores high when it is common in the topic and much rarer elsewhere.

In [22]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

# k-means on the full 768-dim embeddings of all 38,509 comments (not the 2D map)
N_TOPICS = 15
cluster = pd.Series(KMeans(N_TOPICS, n_init=5, random_state=0).fit_predict(emb.to_numpy()), index=emb.index)
rank = {c: k + 1 for k, c in enumerate(cluster.value_counts().index)}   # Topic 1 = largest
topic = cluster.map(rank)

# topic words: the 3 most distinctive words per topic (see above)
vec = CountVectorizer(stop_words="english", min_df=20, binary=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
words = vec.fit_transform(comments.comment_text.loc[emb.index])
vocab = np.array(vec.get_feature_names_out())
overall = np.asarray(words.mean(axis=0)).ravel()
names = {}

for t in sorted(topic.unique()):
    share = np.asarray(words[(topic == t).to_numpy()].mean(axis=0)).ravel()
    score = share * np.log((share + 1e-9) / overall)
    names[t] = ", ".join(vocab[np.argsort(-score)[:3]])

summary = pd.DataFrame({"comments": topic.value_counts().sort_index(), "topic words": pd.Series(names),
                        "mean human toxicity": comments.toxicity.groupby(topic).mean().round(2)})
summary.index.name = "topic"

In [23]:
df["topic"] = topic.loc[df.index].map(lambda t: f"Topic {t}: {names[t]}")
fig = px.scatter(df.reset_index(), x="umap_1", y="umap_2", color="topic", hover_name="id",
                 hover_data={**hover, "topic": True},
                 category_orders={"topic": [f"Topic {t}: {names[t]}" for t in summary.index]},
                 color_discrete_sequence=px.colors.qualitative.Alphabet, opacity=0.6, height=750,
                 title="Comment embeddings (UMAP), colored by topic")
fig.update_traces(marker_size=4)
fig.update_layout(xaxis_visible=False, yaxis_visible=False, plot_bgcolor="white",
                  legend_title_text="Topic (click to hide, double-click to isolate)")
fig.show()